# Validate normal synthetic banking behaviour
This notebook inspects the clean baseline before any fraud typologies are injected.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/raw')
customers = pd.read_parquet(DATA / 'customers.parquet')
accounts = pd.read_parquet(DATA / 'accounts.parquet')
transactions = pd.read_parquet(DATA / 'transactions.parquet')
{name: frame.shape for name, frame in {'customers': customers, 'accounts': accounts, 'transactions': transactions}.items()}

In [ ]:
transactions['amount'].describe(percentiles=[.5, .9, .95, .99]).round(2)

In [ ]:
transactions['channel'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
profile = transactions.merge(customers[['customer_id', 'avg_transaction']], on='customer_id')
profile['amount_vs_customer_average'] = profile['amount'] / profile['avg_transaction']
profile['amount_vs_customer_average'].describe(percentiles=[.5, .9, .95, .99]).round(2)

In [ ]:
assert accounts.customer_id.isin(customers.customer_id).all()
assert transactions.account_id.isin(accounts.account_id).all()
assert transactions.customer_id.isin(customers.customer_id).all()
assert transactions[['balance_before', 'balance_after']].ge(0).all().all()
assert {'is_fraud', 'fraud_rule', 'fraud_scenario'}.isdisjoint(transactions.columns)
print('All baseline checks passed.')